# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- Use Pandas to load, inspect, and clean the dataset appropriately. 
- Transform relevant columns to create measures that address the problem at hand.
- **Conduct EDA: visualization and statistical measures to understand the structure of the data.**
- **Recommend a set of manufacturers to consider as well as specific airplanes conforming to the client's request.**
- **Discuss the relationship between serious injuries/airplane damage incurred and at least *two* factors at play in the incident. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Exploratory Data Analysis  
- Load in the cleaned data

In [ ]:
#Load the data as cleaned_df:
cleaned_df = pd.read_csv("data/AviationDataCleaned.csv")
cleaned_df.head()

## Explore Safety Metrics Across Models/Makes
Remember that the client is interested in separate recommendations for smaller airplanes and larger airplanes. Set the passenger threshold to 20 and separate the plane types. 

In [ ]:
#Let's create a new column called "Airplane.Size" that categorizes planes with
#greater than 20 passengers as "Large" and planes with fewer than 20 passengers
#as "Small":
cleaned_df["Airplane.Size"] = np.where(cleaned_df["Total.Passengers"] > 20, "Large", "Small")
#Check if it worked:
cleaned_df["Airplane.Size"].value_counts()

#### Analyzing Makes

Explore the human injury risk profile for small and large Makes:
- Choose the 15 makes for each group possessing the lowest mean fatal/seriously injured fraction.
- Plot the mean fatal/seriously injured fraction for each of these subgroups side-by-side.

In [ ]:
#First, group the data of Serious.or.Fatal.Fraction first by Airplane.Size, and 
#then by Make:
injury_risk_by_make_and_size = cleaned_df.groupby(["Airplane.Size", "Make"])["Serious.or.Fatal.Fraction"].mean()
injury_risk_by_make_and_size.head(20)

In [ ]:
#Now, find the top 15 small planes with the lowest average injury risk:
small_makes_injury_risk = injury_risk_by_make_and_size["Small"].sort_values().head(15)
small_makes_injury_risk

In [ ]:
#Do the same for the top 15 large planes:
large_makes_injury_risk = injury_risk_by_make_and_size["Large"].sort_values().head(15)
large_makes_injury_risk

In [ ]:
#Create side-by-side subplots:
fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (14, 8))

#Plot small airplane makes:
small_makes_injury_risk.sort_values().plot(kind = "barh", ax = axes[0])

axes[0].set_title("Safest Small Airplane Makes (Safest at Bottom)")
axes[0].set_xlabel("Mean Serious/Fatal Injury Fraction")
axes[0].set_ylabel("Airplane Make")

#Plot large airplane makes:
large_makes_injury_risk.sort_values().plot(kind = "barh", ax = axes[1])

axes[1].set_title("Safest Large Airplane Makes (Safest at Bottom)")
axes[1].set_xlabel("Mean Serious/Fatal Injury Fraction")
axes[1].set_ylabel("Airplane Make")

#Improve spacing:
plt.tight_layout()

#Show plots:
plt.show()

It's virtually impossible for BEECH to have a 100% chance of serious or fatal injury. Likely there are only one or two recorded incidents with BEECH, and every passenger had a serious or fatal injury each time. But this is likely statistically insignificant for BEECH. Looks like MCDONNELL DOUGLAS and BOEING are both pretty safe options for both large and small airplanes. AIRBUS looks safer for small planes than large planes.

**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers serious/fatally injured for small airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
#First, get the ten safest small planes:
top_10_small = small_makes_injury_risk.head(10).index

#Now, create a df that we'll use to make the violin plot:
small_violin_df = cleaned_df[(cleaned_df["Airplane.Size"] == "Small")  & (cleaned_df["Make"].isin(top_10_small))]

#Now, let's make the violin plot:
plt.figure(figsize = (12, 6))
sns.violinplot(data = small_violin_df, x = "Make", y = "Serious.or.Fatal.Fraction")
plt.title("Distribution of Serious/Fatal Injury Fractions for Safest Small Aircraft Makes")
plt.xticks(rotation = 45)
plt.show()

**Distribution of injury rates: large makes**

Use a strip plot to look at the distribution of the fraction of passengers serious/fatally injured for large airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
#First, get the ten safest large planes:
top_10_large = large_makes_injury_risk.head(10).index

#Now, create a df that we'll use to make the strip plot:
large_strip_plot_df = cleaned_df[(cleaned_df["Airplane.Size"] == "Large")  & (cleaned_df["Make"].isin(top_10_large))]

#Now, let's make the strip plot:
plt.figure(figsize = (12, 6))
sns.stripplot(data = large_strip_plot_df, x = "Make", y = "Serious.or.Fatal.Fraction")
#plt.ylim(0, 1)
plt.title("Distribution of Serious/Fatal Injury Fractions for Safest Large Aircraft Makes")
plt.xticks(rotation = 45)
plt.show()

**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.** 

Sort your results and keep the lowest 15.

In [ ]:
#First, group the data of "Aircraft.Destroyed" first by Airplane.Size, and then group by Make:
destroyed_risk_by_make_and_size = cleaned_df.groupby(["Airplane.Size", "Make"])["Aircraft.Destroyed"].mean()
destroyed_risk_by_make_and_size.head(20)

In [ ]:
#Now, find the top 15 small planes with the lowest rate of airplane destruction:
small_makes_destroyed_risk = destroyed_risk_by_make_and_size["Small"].sort_values().head(15)
small_makes_destroyed_risk

In [ ]:
#Do the same for the top 15 large planes:
large_makes_destroyed_risk = destroyed_risk_by_make_and_size["Large"].sort_values().head(15)
large_makes_destroyed_risk

#### Provide a short discussion on your findings for your summary statistics and plots:
- Make any recommendations for Makes here based off of the destroyed fraction and fraction fatally/seriously injured
- Comment on the calculated statistics and any corresponding distributions you have visualized.

### Analyze plane types
- plot the mean fatal/seriously injured fraction for both small and larger planes 
- also provide a distributional plot of your choice for the fatal/seriously injured fraction by airplane type (stripplot, violin, etc)  
- filter ensuring that you have at least ten individual examples in each model/make to average over

**Larger planes**

**Smaller planes**
- for smaller planes, limit your plotted results to the makes with the 10 lowest mean serious/fatal injury fractions

### Discussion of Specific Airplane Types
- Discuss what you have found above regarding passenger fraction seriously/ both small and large airplane models.

### Exploring Other Variables
- Investigate how other variables effect aircraft damage and injury. You must choose **two** factors out of the following but are free to analyze more:

- Weather Condition
- Engine Type
- Number of Engines
- Phase of Flight
- Purpose of Flight

For each factor provide a discussion explaining your analysis with appropriate visualization / data summaries and interpreting your findings.